# Нестандартный синтез: GENDYN, SAWDUST, итерации Ди Шипио, клеточный автомат

Все четыре не думают в гармониках и фильтрах: волна строится по правилам — случайным, комбинаторным, итеративным. Звук уже вшит в выходы; чтобы менять — нужны numpy, scipy, matplotlib (`%pip install numpy scipy matplotlib`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from IPython.display import Audio, display
SR = 22050
plt.rcParams.update({'figure.figsize': (11, 2.6)})

def dc_block(x, r=0.995): return lfilter([1, -1], [1, -r], x)
def fade(x, ms=15):
    n = int(ms * SR / 1000); w = np.ones(len(x)); w[:n] = np.linspace(0, 1, n); w[-n:] = np.linspace(1, 0, n); return x * w
def play(x):
    x = fade(dc_block(np.asarray(x, float))); display(Audio(x / (np.abs(x).max() + 1e-9) * 0.9, rate=SR, normalize=False))
def sgram(x, title='', fmax=6000):
    plt.specgram(dc_block(x), NFFT=1024, Fs=SR, noverlap=896, cmap='Greys', vmin=-110)
    plt.ylim(0, fmax); plt.title(title); plt.xlabel('с'); plt.ylabel('Гц'); plt.show()
def mirror(v, lo, hi):
    '''отражение от стенок — как у Ксенакиса'''
    span = hi - lo
    if span <= 0: return np.full_like(v, lo)
    v = np.mod(v - lo, 2 * span); return lo + np.where(v > span, 2 * span - v, v)

## 1. GENDYN — динамический стохастический синтез (Ксенакис, 1991)

Период — ломаная из n вершин. На каждом новом периоде у каждой вершины **случайно блуждают** длина отрезка и высота точки. Блуждание двухступенчатое: случайный шаг меняет «скорость», скорость меняет значение; обе ограничены зеркальными стенками.

- высота = SR / сумма длин отрезков — она не задаётся, она *получается*;
- узкие стенки длины → устойчивый тон с живым тембром; широкие → вой, свист, гул;
- шаги здесь по распределению Коши, как у Ксенакиса: изредка огромные скачки.

In [ ]:
def gendyn(dur, n=12, dmin=lambda t: 2, dmax=lambda t: 40, astep=0.08, dstep=1.5, seed=0):
    '''dmin/dmax — стенки длины отрезка в сэмплах (функции времени); astep, dstep — размер случайных шагов'''
    rng = np.random.default_rng(seed)
    amp = rng.uniform(-1, 1, n); av = np.zeros(n)
    dur_ = np.full(n, 0.5 * (dmin(0) + dmax(0))); dv = np.zeros(n)
    out = []; total = 0; N = int(dur * SR)
    cauchy = lambda s: np.clip(s * np.tan(np.pi * (rng.random(n) - 0.5)), -8 * s, 8 * s)
    while total < N:
        t = total / SR; lo, hi = dmin(t), dmax(t)
        av = mirror(av + cauchy(astep), -0.3, 0.3); amp = mirror(amp + av, -1, 1)
        dv = mirror(dv + cauchy(dstep), -(hi - lo) / 4 - 0.5, (hi - lo) / 4 + 0.5); dur_ = mirror(dur_ + dv, lo, hi)
        lens = np.maximum(1, np.round(dur_)).astype(int)
        pts = np.append(amp, amp[0])
        for k in range(n):
            out.append(np.linspace(pts[k], pts[k + 1], lens[k], endpoint=False))
        total += lens.sum()
    return np.concatenate(out)[:N]

### 1.1 Свобода: широкие стенки — вой и гул

In [ ]:
x = gendyn(6, n=12, dmin=lambda t: 1, dmax=lambda t: 60, seed=1)
sgram(x, 'GENDYN, длина отрезка 1…60 сэмплов'); play(x)

### 1.2 Хаос сходится в тон: стенки длины сжимаются к 15 сэмплам (≈ 120 Гц при 12 вершинах)

In [ ]:
D = 8
x = gendyn(D, n=12, dmin=lambda t: 15 - 14 * max(0, 1 - t / (D * .7)), dmax=lambda t: 15 + 45 * max(0, 1 - t / (D * .7)), seed=2)
sgram(x, 'стенки сжимаются: хаос -> тон'); play(x)

### 1.3 Тон прибит, блуждает только форма — живой «дышащий» тембр

In [ ]:
x = gendyn(6, n=20, dmin=lambda t: 6, dmax=lambda t: 6, astep=0.03, seed=3)
sgram(x, 'длина фиксирована (≈184 Гц), блуждают амплитуды'); play(x)

### 1.4 Мало вершин против многих: n = 3 (грубо, свист) и n = 40 (шерсть, шум)

In [ ]:
a = gendyn(3, n=3, dmin=lambda t: 5, dmax=lambda t: 80, seed=4)
b = gendyn(3, n=40, dmin=lambda t: 1, dmax=lambda t: 8, seed=5)
x = np.concatenate([fade(a, 40), fade(b, 40)]); sgram(x, 'n = 3, затем n = 40'); play(x)

## 2. SAWDUST (Херберт Брюн, 1976)

Брюн собирал звук из **элементов** — значение амплитуды, удержанное k сэмплов. Элементы складываются в **звенья** (link) — это один период. Дальше звенья не играют, а **превращаются друг в друга**: меняются длины элементов (целые числа сэмплов — отсюда характерная цифровая ступенчатость), звенья смешиваются, чередуются.

Звук — ступенчатый, колючий, «машинный» в лучшем смысле. Ни одного синуса.

In [ ]:
def link(vals, lens): return np.repeat(np.asarray(vals, float), np.asarray(lens, int))

def vary(A, B, dur):
    '''звено A плавно превращается в B: интерполируются значения и (целые) длины элементов'''
    (va, la), (vb, lb) = A, B; out = []; total = 0; N = int(dur * SR)
    while total < N:
        u = total / N
        p = link((1 - u) * np.array(va) + u * np.array(vb), np.maximum(1, np.round((1 - u) * np.array(la) + u * np.array(lb))))
        out.append(p); total += len(p)
    return np.concatenate(out)[:N]

def mingle(A, B, dur, prob=lambda u: u, seed=0):
    '''каждый период — A или B, вероятность B растёт'''
    rng = np.random.default_rng(seed); out = []; total = 0; N = int(dur * SR)
    pa, pb = link(*A), link(*B)
    while total < N:
        p = pb if rng.random() < prob(total / N) else pa; out.append(p); total += len(p)
    return np.concatenate(out)[:N]

A = ([1, -0.4, 0.7, -1, 0.2, -0.6], [20, 7, 13, 25, 4, 30])     # период 99 сэмплов ≈ 223 Гц
B = ([-0.8, 1, -1, 0.5, -0.2, 0.9], [3, 40, 9, 2, 60, 11])     # период 125 ≈ 176 Гц

### 2.1 Одно звено, потом другое

In [ ]:
x = np.concatenate([fade(np.tile(link(*A), 500), 30)[:SR * 2], fade(np.tile(link(*B), 400), 30)[:SR * 2]])
play(x)

### 2.2 vary: A превращается в B за 8 секунд — высота и тембр ползут ступеньками целых сэмплов

In [ ]:
x = vary(A, B, 8); sgram(x, 'vary A -> B'); play(x)

### 2.3 Медленно: звено растягивается в 30 раз — элементы слышны как ритм щелчков, потом снова тон

In [ ]:
Bslow = (B[0], [l * 30 for l in B[1]])
x = np.concatenate([vary(A, Bslow, 5), vary(Bslow, A, 5)]); sgram(x, 'A -> B×30 -> A'); play(x)

### 2.4 mingle: периоды A и B вперемешку, вероятность B 0 → 1 — нерегулярная смесь даёт субгармоники и хрип

In [ ]:
x = mingle(A, B, 7, seed=1); sgram(x, 'mingle A/B'); play(x)

## 3. Итерации функций (Агостино Ди Шипио, 1990-е)

**Functional Iteration Synthesis:** сэмпл = функция, применённая к себе n раз: x → sin(r·x) → sin(r·sin(r·x)) → … Параметр r медленно ведётся осциллятором — так появляется высота. Малое n и r — гладко, почти синус; растёт n или r — тембр крошится, на краю хаоса трещит и поёт.

Второй вариант — **отображение как осциллятор:** логистическое x → r·x·(1−x), одно значение держится k сэмплов. Ведём r через бифуркации: тон → удвоение периода (на октаву вниз) → ещё раз → хаос → «окна» порядка внутри хаоса.

In [ ]:
def fis(dur, f, r0, ra, n, x0=0.5):
    t = np.arange(int(dur * SR)) / SR
    r = r0(t) + ra(t) * np.sin(2 * np.pi * f * t)
    x = np.full_like(t, x0)
    for _ in range(n): x = np.sin(r * x)
    return x

def logistic(dur, r, hold=40, x0=0.3):
    steps = int(dur * SR / hold) + 1; x = x0; out = np.empty(steps)
    for k in range(steps):
        x = r(k * hold / SR) * x * (1 - x); out[k] = x
    return np.repeat(out, hold)[:int(dur * SR)]
c = lambda v: (lambda t: v + 0 * t)

### 3.1 Глубина итераций растёт: n = 1, 2, 4, 8, 16 (по секунде), r вокруг 2.6, f = 110

In [ ]:
x = np.concatenate([fade(fis(1.2, 110, c(2.6), c(0.6), n), 25) for n in (1, 2, 4, 8, 16)])
sgram(x, 'n = 1, 2, 4, 8, 16'); play(x)

### 3.2 r ползёт 1 → 4.5 при n = 6: из синуса — в стеклянный звон — в хаотическую крошку

In [ ]:
D = 8
x = fis(D, 82.5, lambda t: 1 + 3.5 * t / D, c(0.4), 6)
sgram(x, 'r0: 1 -> 4.5'); play(x)

### 3.3 Логистическое отображение: r 2.9 → 4, держим каждое значение 40 сэмплов (551 Гц)

In [ ]:
D = 10
x = logistic(D, lambda t: 2.9 + 1.1 * (t / D) ** 0.6)
sgram(x, 'бифуркации: тон, октава вниз, ещё вниз, хаос, окна'); play(x)

### 3.4 Сидеть на краю: r дрожит около точки накопления 3.5699 — то порядок, то хаос

In [ ]:
D = 8
x = logistic(D, lambda t: 3.5699 + 0.012 * np.sin(2 * np.pi * 0.25 * t), hold=60)
x = x + 0.6 * logistic(D, lambda t: 3.83 + 0.004 * np.sin(2 * np.pi * 0.13 * t), hold=91, x0=0.6)
sgram(x, 'край хаоса + окно периода 3'); play(x)

## 4. Клеточный автомат

Элементарный автомат: строка из W клеток, каждое поколение — по правилу из трёх соседей (правила Вольфрама 0…255). Два способа слышать:

- **волна:** строка поколения и есть форма периода (клетка = ±1, чуть сглажено). Новое поколение — новый тембр, высота стоит;
- **спектр:** клетка k включена — звучит k-я гармоника. Каждое поколение — новый аккорд из натурального ряда. Rule 90 — треугольник Серпинского, rule 30 — хаос, rule 110 — «живые» структуры.

In [ ]:
def ca(rule, W, gens, seed=None):
    if seed is None: row = np.zeros(W, int); row[W // 2] = 1
    else: row = np.random.default_rng(seed).integers(0, 2, W)
    tbl = np.array([(rule >> i) & 1 for i in range(8)]); rows = [row]
    for _ in range(gens - 1):
        row = tbl[4 * np.roll(row, 1) + 2 * row + np.roll(row, -1)]; rows.append(row)
    return np.array(rows)

def ca_wave(rule, f0, dur, gen_per_s=6, W=64, seed=None, smooth=3):
    G = int(dur * gen_per_s) + 2; R = ca(rule, W, G, seed) * 2.0 - 1
    k = np.ones(smooth) / smooth
    out = []; total = 0; N = int(dur * SR)
    while total < N:
        g = min(int(total / SR * gen_per_s), G - 1)
        tbl = np.convolve(np.concatenate([R[g], R[g][:smooth]]), k, 'same')[:W]
        L = int(round(SR / f0)); out.append(np.interp(np.arange(L) * W / L, np.arange(W + 1), np.append(tbl, tbl[0]))); total += L
    return np.concatenate(out)[:N]

def ca_spectrum(rule, f0, dur, gen_per_s=3, H=32, seed=None, tilt=0.7):
    G = int(dur * gen_per_s) + 2; R = ca(rule, H, G, seed).astype(float)
    N = int(dur * SR); t = np.arange(N) / SR
    gpos = t * gen_per_s; g0 = np.floor(gpos).astype(int); a = gpos - g0
    a = np.clip(a * 4, 0, 1)                                     # быстрый перелив на смене поколения
    x = np.zeros(N)
    for h in range(H):
        env = (1 - a) * R[g0, h] + a * R[np.minimum(g0 + 1, G - 1), h]
        if env.any() and f0 * (h + 1) < SR / 2: x += env * np.sin(2 * np.pi * f0 * (h + 1) * t) / (h + 1) ** tilt
    return x

fig, ax = plt.subplots(1, 3, figsize=(11, 3))
for a, r in zip(ax, (30, 90, 110)):
    a.imshow(ca(r, 64, 48, seed=None if r != 110 else 1), cmap='Greys', aspect='auto'); a.set_title(f'rule {r}'); a.axis('off')
plt.show()

### 4.1 Волна: rule 30, 6 поколений в секунду, 110 Гц

In [ ]:
x = ca_wave(30, 110, 6); sgram(x, 'rule 30, волна'); play(x)

### 4.2 Волна: rule 90 из одной клетки — тембр растёт треугольником Серпинского и повторяется

In [ ]:
x = ca_wave(90, 82.5, 8, gen_per_s=8); sgram(x, 'rule 90, волна'); play(x)

### 4.3 Спектр: rule 110 со случайного начала — аккорды из натурального ряда, 3 поколения в секунду

In [ ]:
x = ca_spectrum(110, 55, 10, gen_per_s=3, seed=1); sgram(x, 'rule 110, спектр', fmax=2000); play(x)

### 4.4 Спектр быстро: rule 30, 12 поколений в секунду — уже не аккорды, а ритм-мерцание

In [ ]:
x = ca_spectrum(30, 73.4, 8, gen_per_s=12, H=24); sgram(x, 'rule 30, спектр, 12 пок/с', fmax=2000); play(x)

## Что с этим можно делать

| | ручки | характер | цена в AudioWorklet |
|---|---|---|---|
| **GENDYN** | число вершин, стенки длины (≈ высота и её свобода), размер шагов | живой, никогда не повторяется; от воя до «дышащего» тона | копейки: n точек и линейная интерполяция |
| **SAWDUST** | звенья (значения и длины), vary / mingle, скорость превращения | ступенчатый, колючий, цифровой; при растяжении — ритм | копейки: массив и счётчик |
| **FIS (Ди Шипио)** | глубина n, r, глубина модуляции r | от синуса к стеклу и крошке; очень чувствителен | n синусов на сэмпл — дёшево при n ≤ 8 |
| **Логистика** | r, длина удержания | слышимые бифуркации: октавы вниз → хаос → окна | копейки |
| **Автомат: волна** | правило, скорость поколений, ширина | тембр-калейдоскоп при стоящей высоте | копейки |
| **Автомат: спектр** | правило, скорость, число гармоник | аккорды натурального ряда, при скорости — ритм | H синусов (или одна таблица на поколение) |

Общая черта: у всех «случай/правило» — главная ручка, а не тембр. Тихие режимы есть у каждого: GENDYN со сжатыми стенками, FIS с малым n, автомат-спектр медленно — они ближе к дрону; остальное — громкое и шершавое, его стоит держать на краю.